# Heaton Data Preprocessing

The purpose of this file is to preprocess and prepare the rdata from the heaton MODOS satellite temperature dataset, to create a model for it and compare against the results from the MuyGPs paper.

## Import Libraries and data

In [138]:
#first download needed libraries
import pandas as pd
import numpy as np
import sys
import os
import pyreadr

from scipy import stats
from sklearn.preprocessing import MinMaxScaler
import plotly.express as px
import plotly.io as pio

from MuyGPyS.neighbors import NN_Wrapper
from MuyGPyS.gp import MuyGPS
from MuyGPyS.gp.deformation import Isotropy, l2
from MuyGPyS.gp.hyperparameter import AnalyticScale, Parameter
from MuyGPyS.gp.kernels import Matern
from MuyGPyS.gp.noise import HomoscedasticNoise
from MuyGPyS.optimize import Bayes_optimize
from MuyGPyS.optimize.batch import sample_batch
from MuyGPyS.optimize.loss import lool_fn
from MuyGPyS._test.sampler import print_results


In [139]:
#read in the base data and display first few rows.
BASE_DIR = os.path.abspath("..") + "\\"
print(pyreadr.list_objects(BASE_DIR + 'data\\AllSatelliteTemps.RData'))
result = pyreadr.read_r(BASE_DIR + 'data\\AllSatelliteTemps.RData')
heaton_df = result["all.sat.temps"]
print(heaton_df.head())
heaton_df.describe()

[{'object_name': 'all.sat.temps', 'columns': ['Lon', 'Lat', 'MaskTemp', 'TrueTemp']}]
         Lon        Lat  MaskTemp  TrueTemp
0 -95.911530  37.068111       NaN       NaN
1 -95.902256  37.068111       NaN       NaN
2 -95.892982  37.068111       NaN       NaN
3 -95.883708  37.068111       NaN       NaN
4 -95.874434  37.068111       NaN       NaN


,Lon,Lat,MaskTemp,TrueTemp
count,150000.000000,150000.000000,105569.000000,148309.000000
mean,-93.597670,35.681652,44.538694,45.124661
std,1.338586,0.803148,3.971667,4.069270
min,-95.911530,34.295192,24.370000,24.370000
25%,-94.754600,34.988422,42.050000,42.370000
50%,-93.597670,35.681652,44.790000,45.350000
75%,-92.440740,36.374881,47.430000,48.410000
max,-91.283811,37.068111,55.410000,55.410000


## Cleaning

In [140]:
#first remove all rows that are missing from the complete dataset. these don't have known readings.
present_readings_df = heaton_df.dropna(subset=["TrueTemp"])
print(present_readings_df.describe())
present_readings_df

                 Lon            Lat       MaskTemp       TrueTemp
count  148309.000000  148309.000000  105569.000000  148309.000000
mean      -93.598122      35.682554      44.538694      45.124661
std         1.330424       0.797419       3.971667       4.069270
min       -95.911530      34.295192      24.370000      24.370000
25%       -94.743008      34.990740      42.050000      42.370000
50%       -93.602307      35.686289      44.790000      45.350000
75%       -92.452333      36.372563      47.430000      48.410000
max       -91.283811      37.068111      55.410000      55.410000


,Lon,Lat,MaskTemp,TrueTemp
6,-95.855886,37.068111,42.39,42.39
7,-95.846612,37.068111,41.85,41.85
8,-95.837338,37.068111,42.53,42.53
9,-95.828064,37.068111,45.27,45.27
10,-95.818790,37.068111,46.09,46.09
...,...,...,...,...
149995,-91.320907,34.295192,30.53,30.53
149996,-91.311633,34.295192,31.17,31.17
149997,-91.302359,34.295192,32.09,32.09
149998,-91.293085,34.295192,32.45,32.45


## Normalizing

In [141]:
#need to scale the coordinates together, because they need to scale the same to keep their relationship.
#de-mean them first so they aren't all clustered on opposite ends.
scaled_df = present_readings_df.copy()
lat_mean = scaled_df["Lat"].mean()
lon_mean = scaled_df["Lon"].mean()
scaled_df["Lon"] = scaled_df["Lon"] - lon_mean
scaled_df["Lat"] = scaled_df["Lat"] - lat_mean
scale_min = min(scaled_df["Lon"].min(), scaled_df["Lat"].min())
scale_max = max(scaled_df["Lon"].max(), scaled_df["Lat"].max())
scale_dif = scale_max - scale_min
print(f"Min: {scale_min}\tMax: {scale_max}\tDifference: {scale_dif}")
scaled_df["Lon"] = ((scaled_df["Lon"] - scale_min) / scale_dif)
scaled_df["Lat"] = ((scaled_df["Lat"] - scale_min) / scale_dif)
scaled_df.describe()

Min: -2.3134083493396673	Max: 2.3143109917779157	Difference: 4.627719341117583


,Lon,Lat,MaskTemp,TrueTemp
count,148309.000000,148309.000000,105569.000000,148309.000000
mean,0.499902,0.499902,44.538694,45.124661
std,0.287490,0.172314,3.971667,4.069270
min,0.000000,0.200109,24.370000,24.370000
25%,0.252505,0.350409,42.050000,42.370000
50%,0.498998,0.500709,44.790000,45.350000
75%,0.747495,0.649006,47.430000,48.410000
max,1.000000,0.799306,55.410000,55.410000


In [142]:
#seperate the training data by keeping the locations that werer included in the masked temperatures.
heaton_train_df = scaled_df.dropna(subset="MaskTemp")[["Lon", "Lat", "TrueTemp"]].copy()
#seperate training data into input and output.
heaton_train_x_df = heaton_train_df[["Lon", "Lat"]].copy()
heaton_train_y_df = heaton_train_df[["TrueTemp"]].copy()
heaton_train_y_df.values.flatten()
print(heaton_train_y_df.shape)
print(f"Heaton training x: {heaton_train_x_df.shape}")
print(f"Heaton training y: {heaton_train_y_df.shape}")

heaton_test_df = scaled_df[scaled_df['MaskTemp'].isna()].copy()
heaton_test_x_df = heaton_test_df[["Lon", "Lat"]].copy()
heaton_test_y_df = heaton_test_df[["TrueTemp"]].copy()
print(f"Heaton testing x: {heaton_test_x_df.shape}")
print(f"Heaton testing y: {heaton_test_y_df.shape}")

(105569, 1)
Heaton training x: (105569, 2)
Heaton training y: (105569, 1)
Heaton testing x: (42740, 2)
Heaton testing y: (42740, 1)


In [143]:
#demean the data
temp_mean = heaton_train_y_df.to_numpy().mean()
print(f"mean temperature: {temp_mean}")
heaton_train_y_demeaned_df = heaton_train_y_df - temp_mean
print(heaton_train_y_df.describe())
heaton_train_y_demeaned_df.describe()

mean temperature: 44.53869402949732
            TrueTemp
count  105569.000000
mean       44.538694
std         3.971667
min        24.370000
25%        42.050000
50%        44.790000
75%        47.430000
max        55.410000


,TrueTemp
count,1.055690e+05
mean,3.652831e-15
std,3.971667e+00
min,-2.016869e+01
25%,-2.488694e+00
50%,2.513060e-01
75%,2.891306e+00
max,1.087131e+01


In [144]:
#convert everything into numpy arrays
heaton_train_x = heaton_train_x_df.to_numpy() 
heaton_train_y = heaton_train_y_demeaned_df["TrueTemp"].to_numpy() 
heaton_test_x = heaton_test_x_df.to_numpy()
heaton_test_y = heaton_test_y_df["TrueTemp"].to_numpy()

# Model Creation

## Nearest Neighbor and Batches

In [145]:
nn_count = 50
nbrs_lookup = NN_Wrapper(heaton_train_x, nn_count, nn_method="exact", algorithm="ball_tree")

In [146]:
batch_count = 2000
batch_indices, batch_nn_indices = sample_batch(
    nbrs_lookup, batch_count, len(heaton_train_x)
)

## Setting and optimizing hyperparameters

In [147]:
heaton_muygps = MuyGPS(
    kernel=Matern(
        smoothness=Parameter("log_sample", (0.1, 5.0)),
        deformation=Isotropy(
            l2,
            length_scale=Parameter(1.0),
        ),
    ),
    noise=HomoscedasticNoise(0.001),
    scale=AnalyticScale(),
)

In [148]:
(
    batch_crosswise_dists,
    batch_pairwise_dists,
    batch_ys,
    batch_nn_ys,
) = heaton_muygps.make_train_tensors(
    batch_indices,
    batch_nn_indices,
    heaton_train_x,
    heaton_train_y,
)

In [149]:
heaton_muygps_optimized = Bayes_optimize(
    heaton_muygps,
    batch_ys,
    batch_nn_ys,
    batch_crosswise_dists,
    batch_pairwise_dists,
    loss_fn=lool_fn,
    verbose=True,
    random_state=42,
    init_points=20,
    n_iter=30,
)

parameters to be optimized: ['smoothness']
bounds: [[0.1 5. ]]
initial x0: [0.67896521]
|   iter    |  target   | smooth... |
-------------------------------------
| 1         | -2309.785 | 0.6789652 |
| 2         | -80543.09 | 1.9352465 |
| 3         | -76731.09 | 4.7585001 |
| 4         | -77353.88 | 3.6867703 |
| 5         | -78012.03 | 3.0334265 |
| 6         | -14008.87 | 0.8644913 |
| 7         | -13996.08 | 0.8643731 |
| 8         | 115.82689 | 0.3846096 |
| 9         | -76928.52 | 4.3442631 |
| 10        | -77996.63 | 3.0454635 |
| 11        | -77450.07 | 3.5695556 |
| 12        | -401.3119 | 0.2008640 |
| 13        | -76691.74 | 4.8525582 |
| 14        | -77020.38 | 4.1789689 |
| 15        | -59367.86 | 1.1404616 |
| 16        | -32560.39 | 0.9909423 |
| 17        | -33959.79 | 0.9986820 |
| 18        | -81603.22 | 1.5907869 |
| 19        | -78561.26 | 2.6713065 |
| 20        | -79599.85 | 2.2165305 |
| 21        | -81477.21 | 1.5270227 |
| 22        | -1043.466 | 0.1       |


In [150]:
heaton_muygps_optimized2 = heaton_muygps_optimized.optimize_scale(
    batch_pairwise_dists,
    batch_nn_ys
)

## Inference

In [151]:
test_count = heaton_test_x.shape[0]
test_indices = np.arange(test_count)
test_nn_indices, _ = nbrs_lookup.get_nns(heaton_test_x)

In [152]:
(
    test_crosswise_dists,
    test_pairwise_dists,
    test_nn_ys,
) = heaton_muygps.make_predict_tensors(
    test_indices,
    test_nn_indices,
    heaton_test_x,
    heaton_train_x,
    heaton_train_y,
)

In [153]:
kcross = heaton_muygps_optimized.kernel(test_crosswise_dists)
kin = heaton_muygps_optimized.kernel(test_pairwise_dists)

In [154]:
predictions = heaton_muygps_optimized.posterior_mean(kin, kcross, test_nn_ys)
predictions = predictions + temp_mean
variances = heaton_muygps_optimized.posterior_variance(kin, kcross)
confidence_intervals = np.sqrt(variances) * 1.96
coverage = np.count_nonzero(np.abs(heaton_test_y - predictions) < confidence_intervals) / test_count

In [155]:
print_results(
    heaton_test_y, ("optimized", heaton_muygps_optimized, predictions, variances, confidence_intervals, coverage)
)

name,smoothness,length scale,noise variance,variance scale,rmse,mean variance,mean confidence interval,coverage
optimized,0.442561,1.000000,0.001000,104.541124,1.681639,2.318016,2.678401,0.917829


Results for the RMSE and Coverage match the performance of the model specified in table 1 of the MuyGPs paper, using a length scale of 1.0 and the u1 mean function. This model was made with the same general pipeline used to create the AQI model, meaning any issues with AQI model performance are the fault of the data and / or hyperparameters chosen.